# FastSpeech 2 — Non-Autoregressive TTS

**Paper:** *FastSpeech 2: Fast and High-Quality End-to-End Text to Speech* (Ren et al., 2021)
arXiv: [2006.04558](https://arxiv.org/abs/2006.04558)

---

## Motivation

Tacotron 2 is **autoregressive**: each mel frame depends on the previous one.
This causes two problems:

| Problem | Impact |
|---------|--------|
| Sequential decoding | Slow inference — cannot parallelize |
| Attention collapse | Sometimes skips or repeats words |

**FastSpeech 2** solves both by predicting **all mel frames in parallel**, using a duration predictor to align text with audio.

### Architecture Overview

```
Text -> Text Encoder (FFT Blocks)
         |
    Variance Adaptor
    |- Duration Predictor  -> expand hidden states
    |- Pitch Predictor     -> add F0 embedding
    +- Energy Predictor    -> add energy embedding
         |
   Mel Decoder (FFT Blocks)
         |
   Mel Spectrogram -> Vocoder -> Waveform
```

The key insight: replace the fragile attention mechanism with an explicit **duration predictor** trained on ground-truth durations extracted from Montreal Forced Aligner (MFA).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
print("PyTorch:", torch.__version__)

## 1. Feed-Forward Transformer (FFT) Block

The FFT Block is the core building block of FastSpeech 2.
It replaces the RNN with **multi-head self-attention + 1D convolution FFN**, enabling full parallelism.

```
Input
  +- LayerNorm -> Multi-Head Self-Attention -> Residual
       +- LayerNorm -> Conv1D -> ReLU -> Conv1D -> Residual
Output
```

In [ ]:
class FFTBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, kernel_size=9, dropout=0.1):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff_c1 = nn.Conv1d(d_model, d_ff, kernel_size, padding=kernel_size // 2)
        self.ff_c2 = nn.Conv1d(d_ff, d_model, kernel_size, padding=kernel_size // 2)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # x: (B, T, d_model)
        r = x
        x = self.norm1(x)
        x, _ = self.attn(x, x, x, key_padding_mask=mask)
        x = r + self.drop(x)

        r = x
        x = self.norm2(x)
        x = F.relu(self.ff_c1(x.transpose(1, 2)))
        x = self.ff_c2(x).transpose(1, 2)
        x = r + self.drop(x)
        return x

# Quick test
block = FFTBlock(d_model=256, n_heads=2, d_ff=1024)
dummy = torch.randn(2, 20, 256)
out = block(dummy)
print("FFTBlock output:", out.shape)  # (2, 20, 256)

## 2. Duration Predictor

The Duration Predictor takes encoder hidden states and outputs **one positive number per phoneme**: how many mel frames that phoneme spans.

During **training**: ground-truth durations come from MFA forced alignment.
During **inference**: predicted durations are rounded to integers for length regulation.

```
hidden -> Conv1D-ReLU-LN -> Conv1D-ReLU-LN -> Linear -> log-duration
```

Loss: MSE on **log-scale** durations (log makes short and long durations equally weighted).

In [ ]:
class DurationPredictor(nn.Module):
    def __init__(self, d_model, d_hidden=256, kernel_size=3, dropout=0.1):
        super().__init__()
        self.c1   = nn.Conv1d(d_model,   d_hidden, kernel_size, padding=kernel_size // 2)
        self.c2   = nn.Conv1d(d_hidden,  d_hidden, kernel_size, padding=kernel_size // 2)
        self.ln1  = nn.LayerNorm(d_hidden)
        self.ln2  = nn.LayerNorm(d_hidden)
        self.drop = nn.Dropout(dropout)
        self.proj = nn.Linear(d_hidden, 1)

    def forward(self, x):
        # x: (B, T_text, d_model)
        h = F.relu(self.c1(x.transpose(1, 2))).transpose(1, 2)   # (B, T, d_hidden)
        h = self.drop(self.ln1(h))
        h = F.relu(self.c2(h.transpose(1, 2))).transpose(1, 2)
        h = self.drop(self.ln2(h))
        return self.proj(h).squeeze(-1)   # (B, T_text)

dp = DurationPredictor(d_model=256)
enc_out = torch.randn(2, 15, 256)
log_durations = dp(enc_out)
print("Predicted log-durations:", log_durations.shape)   # (2, 15)
durations = torch.clamp(log_durations.exp().round().long(), min=1)
print("Sample durations:", durations[0].tolist())

## 3. Length Regulator

The Length Regulator **repeats each phoneme hidden state** according to its duration, converting the phoneme sequence into a mel-frame sequence.

```
Phoneme:  [p]  [ae]  [t]
Duration: [ 3]  [ 5]  [ 2]
            |
Frames:   [p][p][p][ae][ae][ae][ae][ae][t][t]
```

This is the **alignment-free** replacement for attention.

In [ ]:
def length_regulate(x, durations):
    # x:         (B, T_text, d_model)
    # durations: (B, T_text) integer frame counts per phoneme
    # Returns:   (B, T_mel, d_model)
    outputs = []
    for b in range(x.size(0)):
        repeated = torch.repeat_interleave(x[b], durations[b], dim=0)
        outputs.append(repeated)
    max_len = max(o.size(0) for o in outputs)
    padded = torch.zeros(len(outputs), max_len, x.size(-1))
    for b, o in enumerate(outputs):
        padded[b, :o.size(0)] = o
    return padded

# Demo
hidden = torch.randn(2, 5, 256)
durs = torch.tensor([[3, 2, 4, 1, 3], [2, 3, 2, 4, 2]])
regulated = length_regulate(hidden, durs)
print("Phonemes: 5  ->  Mel frames:", regulated.shape)
print("Total frames batch-0:", durs[0].sum().item(),
      "  batch-1:", durs[1].sum().item())

# Visualize the expansion
fig, ax = plt.subplots(figsize=(10, 2))
ph_labels = ["p", "ae", "t", "ax", "n"]
dur_vals  = durs[0].tolist()
colors    = plt.cm.Set2.colors
x_pos = 0
for i, (ph, d) in enumerate(zip(ph_labels, dur_vals)):
    ax.barh(0, d, left=x_pos, height=0.5, color=colors[i % len(colors)], edgecolor="k")
    ax.text(x_pos + d/2, 0, f"{ph}\n({d})", ha="center", va="center", fontsize=11)
    x_pos += d
ax.set_xlim(0, x_pos)
ax.set_yticks([])
ax.set_xlabel("Mel Frame Index")
ax.set_title("Length Regulation: each phoneme repeated by its duration")
plt.tight_layout()
plt.savefig("figures/length_regulate.png", dpi=110, bbox_inches="tight")
plt.show()
print("Saved figures/length_regulate.png")

## 4. Pitch & Energy Predictors (Variance Adaptor)

FastSpeech 2 adds two additional **variance predictors** on the length-regulated sequence:

| Predictor | Target | Embedding |
|-----------|--------|-----------|
| **Pitch** | F0 (fundamental frequency) per frame | Quantized into 256 bins -> learned embedding |
| **Energy** | L2 norm of mel frame | Quantized into 256 bins -> learned embedding |

Both are added to the length-regulated hidden states before the mel decoder, giving the model **explicit prosody control** at inference time.

In [ ]:
class VariancePredictor(nn.Module):
    def __init__(self, d_model, d_hidden=256, kernel_size=3, dropout=0.1):
        super().__init__()
        self.c1   = nn.Conv1d(d_model,  d_hidden, kernel_size, padding=kernel_size // 2)
        self.c2   = nn.Conv1d(d_hidden, d_hidden, kernel_size, padding=kernel_size // 2)
        self.ln1  = nn.LayerNorm(d_hidden)
        self.ln2  = nn.LayerNorm(d_hidden)
        self.drop = nn.Dropout(dropout)
        self.proj = nn.Linear(d_hidden, 1)

    def forward(self, x):
        h = F.relu(self.c1(x.transpose(1, 2))).transpose(1, 2)
        h = self.drop(self.ln1(h))
        h = F.relu(self.c2(h.transpose(1, 2))).transpose(1, 2)
        h = self.drop(self.ln2(h))
        return self.proj(h).squeeze(-1)   # (B, T_mel)

class VarianceAdaptor(nn.Module):
    def __init__(self, d_model, n_bins=256):
        super().__init__()
        self.duration_predictor = DurationPredictor(d_model)
        self.pitch_predictor    = VariancePredictor(d_model)
        self.energy_predictor   = VariancePredictor(d_model)
        self.pitch_embed  = nn.Embedding(n_bins, d_model)
        self.energy_embed = nn.Embedding(n_bins, d_model)

    def forward(self, x, durations=None):
        log_dur = self.duration_predictor(x)
        if durations is None:
            durations = torch.clamp(log_dur.exp().round().long(), min=1)
        x = length_regulate(x, durations)
        pitch  = self.pitch_predictor(x)
        energy = self.energy_predictor(x)
        p_idx = torch.clamp((pitch  * 10 + 128).long(), 0, 255)
        e_idx = torch.clamp((energy * 10 + 128).long(), 0, 255)
        x = x + self.pitch_embed(p_idx) + self.energy_embed(e_idx)
        return x, log_dur

va = VarianceAdaptor(d_model=256)
enc_out = torch.randn(2, 10, 256)
gt_durs = torch.tensor([[3,2,4,1,3,2,3,2,1,4],[2,3,2,4,2,3,1,2,3,2]])
out, log_dur = va(enc_out, gt_durs)
print("VarianceAdaptor output:", out.shape)
print("Log durations:", log_dur.shape)

## 5. Full FastSpeech 2 Model

Putting it all together:

```
Phoneme IDs -> Embedding -> Text Encoder (N x FFT Blocks)
                                  |
                          Variance Adaptor
                   (Duration + Pitch + Energy)
                                  |
                     Mel Decoder (N x FFT Blocks)
                                  |
                         Linear -> Mel Spectrogram
```

Training losses (all combined):
- **L_mel**: MAE between predicted and GT mel
- **L_dur**: MSE on log-durations
- **L_pitch**: MSE on pitch per frame
- **L_energy**: MSE on energy per frame

In [ ]:
class FastSpeech2(nn.Module):
    def __init__(self,
                 vocab_size=100,
                 d_model=256,
                 n_heads=2,
                 d_ff=1024,
                 n_enc_layers=4,
                 n_dec_layers=4,
                 n_mels=80,
                 n_bins=256):
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_enc = nn.Embedding(2000, d_model)
        self.encoder = nn.ModuleList([FFTBlock(d_model, n_heads, d_ff) for _ in range(n_enc_layers)])
        self.va      = VarianceAdaptor(d_model, n_bins)
        self.decoder = nn.ModuleList([FFTBlock(d_model, n_heads, d_ff) for _ in range(n_dec_layers)])
        self.mel_proj = nn.Linear(d_model, n_mels)

    def forward(self, phoneme_ids, durations=None):
        B, T = phoneme_ids.shape
        pos = torch.arange(T, device=phoneme_ids.device).unsqueeze(0)
        x = self.embed(phoneme_ids) + self.pos_enc(pos)
        for layer in self.encoder:
            x = layer(x)
        x, log_dur = self.va(x, durations)
        T_mel = x.size(1)
        pos_m = torch.arange(T_mel, device=x.device).unsqueeze(0)
        x = x + self.pos_enc(pos_m)
        for layer in self.decoder:
            x = layer(x)
        mel = self.mel_proj(x)
        return mel, log_dur

model = FastSpeech2(vocab_size=100, n_enc_layers=2, n_dec_layers=2)
phonemes = torch.randint(1, 100, (2, 10))
gt_durs  = torch.tensor([[3,2,4,1,3,2,3,2,1,4],[2,3,2,4,2,3,1,2,3,2]])
mel_out, log_dur_out = model(phonemes, gt_durs)
print("Mel output:", mel_out.shape)
print("Log dur:", log_dur_out.shape)
print(f"Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 6. Training Loss & One Step

In [ ]:
def fastspeech2_loss(mel_pred, mel_gt, log_dur_pred, log_dur_gt):
    T = min(mel_pred.size(1), mel_gt.size(1))
    l_mel = F.l1_loss(mel_pred[:, :T], mel_gt[:, :T])
    l_dur = F.mse_loss(log_dur_pred, log_dur_gt)
    return l_mel + l_dur, l_mel, l_dur

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
T_mel_gt  = gt_durs.sum(dim=1).max().item()
gt_mel    = torch.randn(2, T_mel_gt, 80)
gt_log_dur = torch.log(gt_durs.float().clamp(min=1))

optimizer.zero_grad()
mel_pred, log_dur_pred = model(phonemes, gt_durs)
loss, l_mel, l_dur = fastspeech2_loss(mel_pred, gt_mel, log_dur_pred, gt_log_dur)
loss.backward()
optimizer.step()
print(f"Total loss: {loss.item():.4f}  (mel={l_mel.item():.4f}, dur={l_dur.item():.4f})")

## 7. Inference: Prosody Control

At inference time you can **scale** the predicted durations, pitch, and energy to control speaking rate and expressiveness — without retraining.

In [ ]:
model.eval()
with torch.no_grad():
    phonemes_inf = torch.randint(1, 100, (1, 8))
    mel_normal, _ = model(phonemes_inf)
    print("Normal mel shape:", mel_normal.shape)

# Manual duration scaling (1.5x slower)
with torch.no_grad():
    B, T = phonemes_inf.shape
    pos = torch.arange(T).unsqueeze(0)
    x = model.embed(phonemes_inf) + model.pos_enc(pos)
    for layer in model.encoder:
        x = layer(x)
    log_dur = model.va.duration_predictor(x)
    durs_slow = torch.clamp((log_dur.exp() * 1.5).round().long(), min=1)
    durs_norm = torch.clamp( log_dur.exp()        .round().long(), min=1)
    print(f"Normal total frames:  {durs_norm.sum().item()}")
    print(f"Slow   total frames:  {durs_slow.sum().item()}  (1.5x duration scaling)")

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
with torch.no_grad():
    mel_slow, _ = model(phonemes_inf, durs_slow)
for ax, mel, title in zip(axes,
                           [mel_normal[0].T, mel_slow[0].T],
                           ["Normal speed", "Slow (1.5x duration)"]):
    ax.imshow(mel.numpy(), origin="lower", aspect="auto", cmap="magma")
    ax.set_title(title)
    ax.set_xlabel("Frame")
    ax.set_ylabel("Mel bin")
plt.tight_layout()
plt.savefig("figures/fs2_prosody.png", dpi=110, bbox_inches="tight")
plt.show()

## Summary

| Component | Role |
|-----------|------|
| **FFT Block** | Parallel self-attention + Conv1D FFN |
| **Duration Predictor** | Phoneme -> frame count (replaces attention) |
| **Length Regulator** | Repeat hidden states by duration |
| **Pitch Predictor** | Per-frame F0 control |
| **Energy Predictor** | Per-frame loudness control |

FastSpeech 2 produces **comparable quality to Tacotron 2** while being ~38x faster at inference, and allows fine-grained prosody control without retraining.

**Next:** VITS — end-to-end TTS that skips the separate vocoder entirely.